# Notebook: entrenar YOLOX desde ZIP COCO, exportar a ONNX y descargar el modelo

Este notebook esta pensado para usuarios de Google Colab que quieran entrenar un modelo YOLOX con su propio dataset en formato COCO, exportarlo a ONNX y descargar el archivo final listo para usar.


## Antes de empezar

Este notebook hace todo el flujo completo:

1. instala YOLOX y sus dependencias,
2. recibe un archivo `.zip` con tu dataset,
3. reorganiza el dataset dentro de Colab,
4. entrena el modelo,
5. exporta el mejor checkpoint a ONNX,
6. inserta los nombres de clase dentro del metadato del ONNX,
7. descarga el ONNX final.

Recomendaciones:

- Ejecuta las celdas en orden, de arriba hacia abajo.
- Usa un entorno de Colab con GPU activa.
- Sube un unico archivo `.zip` por ejecucion.
- No cambies nombres de carpetas ni de archivos JSON si no sabes exactamente que estas haciendo.


## Estructura esperada del dataset ZIP

Este notebook fue preparado y probado con una estructura como esta:

```text
dataset_.coco.zip
|-- test/
|   |-- _annotations.coco.json
|   |-- img1.jpg
|   `-- img2.jpg
|-- train/
|   |-- _annotations.coco.json
|   |-- img1.jpg
|   `-- img2.jpg
`-- valid/
    |-- _annotations.coco.json
    |-- img3.jpg
    `-- img4.jpg
```

Reglas importantes para que el notebook detecte bien tu dataset:

- Debe existir al menos una carpeta con imagenes y al menos un archivo `.json` en formato COCO.
- Las imagenes pueden ser `.jpg`, `.jpeg` o `.png`.
- El JSON de entrenamiento debe quedar dentro de `train/` o tener `train` en el nombre.
- El JSON de validacion debe quedar dentro de `valid/` o tener `val` o `valid` en el nombre.
- Si existe `test/`, se ignora para el entrenamiento.
- El campo `file_name` dentro del JSON debe coincidir exactamente con el nombre real de cada imagen.


## Requisitos minimos del JSON COCO

Cada archivo COCO debe incluir estas claves:

- `images`
- `annotations`
- `categories`

Ademas:

- `categories` define las clases del modelo.
- El numero de clases se calcula automaticamente a partir de `categories`.
- El ONNX final guardara esos nombres de clase dentro del metadato `names`.
- Si `train` y `valid` no estan bien separados en tu ZIP, el entrenamiento y la validacion pueden quedar mal configurados.


In [ ]:
# ======================================
# 1) CONFIGURACION GENERAL DEL NOTEBOOK
# ======================================
from pathlib import Path

BASE_DIR = Path("/content")
YOLOX_DIR = BASE_DIR / "YOLOX"
DATASET_RAW_DIR = BASE_DIR / "dataset_raw"
DATASET_DIR = BASE_DIR / "dataset_coco"
ANN_DIR = DATASET_DIR / "annotations"
EXP_FILE = YOLOX_DIR / "yolox_config.py"

YOLOX_VARIANT = "yolox_s"   # yolox_nano, yolox_tiny, yolox_s, yolox_m, yolox_l, yolox_x
BATCH_SIZE = 8
MAX_EPOCH = 30
# INPUT_WIDTH e INPUT_HEIGHT deben ser multiplos de 32 para YOLOX.
INPUT_WIDTH = 640
INPUT_HEIGHT = 640

if INPUT_WIDTH % 32 != 0 or INPUT_HEIGHT % 32 != 0:
    raise ValueError("INPUT_WIDTH e INPUT_HEIGHT deben ser multiplos de 32 para YOLOX")
NUM_WORKERS = 2
USE_FP16 = True

PRETRAINED_WEIGHTS = {
    "yolox_nano": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth",
    "yolox_tiny": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth",
    "yolox_s":    "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth",
    "yolox_m":    "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_m.pth",
    "yolox_l":    "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_l.pth",
    "yolox_x":    "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_x.pth",
}

MODEL_SCALES = {
    "yolox_nano": (0.33, 0.25),
    "yolox_tiny": (0.33, 0.375),
    "yolox_s":    (0.33, 0.50),
    "yolox_m":    (0.67, 0.75),
    "yolox_l":    (1.00, 1.00),
    "yolox_x":    (1.33, 1.25),
}

assert YOLOX_VARIANT in PRETRAINED_WEIGHTS, f"Variante no soportada: {YOLOX_VARIANT}"
depth, width = MODEL_SCALES[YOLOX_VARIANT]
PRETRAINED_PATH = BASE_DIR / f"{YOLOX_VARIANT}.pth"
PRETRAINED_URL = PRETRAINED_WEIGHTS[YOLOX_VARIANT]

print("Configuracion actual")
print("YOLOX_VARIANT =", YOLOX_VARIANT)
print("BATCH_SIZE    =", BATCH_SIZE)
print("MAX_EPOCH     =", MAX_EPOCH)
print("INPUT_WIDTH   =", INPUT_WIDTH)
print("INPUT_HEIGHT  =", INPUT_HEIGHT)
print("NUM_WORKERS   =", NUM_WORKERS)
print("USE_FP16      =", USE_FP16)
print("PRETRAINED    =", PRETRAINED_PATH)
print("depth,width   =", (depth, width))


In [ ]:
# =========================================
# 2) VERIFICAR QUE COLAB TENGA GPU REAL
# =========================================
import torch, subprocess

print("torch.__version__ =", torch.__version__)
print("torch.cuda.is_available() =", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    raise RuntimeError(
        "Colab no tiene GPU activa. Ve a Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU."
    )


In [ ]:
# ==================================
# 3) INSTALAR DEPENDENCIAS Y YOLOX
# ==================================
%cd /content

from pathlib import Path
import re
import shutil

YOLOX_DIR = Path("/content/YOLOX")

!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q pycocotools onnx onnxscript onnxruntime loguru tabulate thop ninja "jedi>=0.16"

# Borrar YOLOX viejo para evitar requirements danados de ejecuciones anteriores
if YOLOX_DIR.exists():
    shutil.rmtree(YOLOX_DIR)
    print("YOLOX anterior eliminado")

!git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

req_file = YOLOX_DIR / "requirements.txt"
txt = req_file.read_text(encoding="utf-8")

# Quitar simplificacion ONNX porque en Colab reciente suele dar problemas de build
txt = re.sub(r'^.*onnx-simplifier.*\n?', '', txt, flags=re.MULTILINE)
txt = re.sub(r'^.*onnxsim.*\n?', '', txt, flags=re.MULTILINE)

req_file.write_text(txt, encoding="utf-8")
print("requirements.txt limpio")

!python -m pip install -q -r /content/YOLOX/requirements.txt

print("Instalacion terminada.")


In [ ]:
# ======================================================
# 4) FORZAR PYTHONPATH Y VALIDAR IMPORTACION DE YOLOX
# ======================================================
import os, sys, torch

if str(YOLOX_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOX_DIR))

current_pythonpath = os.environ.get("PYTHONPATH", "")
paths = [p for p in current_pythonpath.split(":") if p] if current_pythonpath else []
if str(YOLOX_DIR) not in paths:
    os.environ["PYTHONPATH"] = f"{YOLOX_DIR}:{current_pythonpath}" if current_pythonpath else str(YOLOX_DIR)

print("PYTHONPATH =", os.environ.get("PYTHONPATH"))
print("torch.cuda.is_available() =", torch.cuda.is_available())

import yolox
print("Importacion OK de yolox desde:", yolox.__file__)


In [ ]:
# ============================================
# 5) DESCARGAR EL PESO BASE PREENTRENADO
# ============================================
if not PRETRAINED_PATH.exists():
    !wget -O {PRETRAINED_PATH} {PRETRAINED_URL}
else:
    print("Ya existe:", PRETRAINED_PATH)

print("Peso base listo en:", PRETRAINED_PATH)


In [ ]:
# =====================
# 6) SUBIR ZIP COCO
# =====================
from google.colab import files

print("Sube un unico archivo .zip con tu dataset en formato COCO.")
print("Estructura recomendada:")
print("- train/_annotations.coco.json + imagenes")
print("- valid/_annotations.coco.json + imagenes")
print("- test/ es opcional y se ignora para entrenar")
print("- Extensiones soportadas: .jpg, .jpeg, .png")

uploaded = files.upload()

zip_candidates = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
if not zip_candidates:
    raise FileNotFoundError("No se subio ningun archivo .zip")
if len(zip_candidates) > 1:
    raise ValueError("Se detectaron varios ZIP. Sube solo un archivo por ejecucion.")

ZIP_PATH = BASE_DIR / zip_candidates[0]
print("ZIP detectado:", ZIP_PATH)


In [ ]:
# ===========================================
# 7) EXTRAER ZIP Y REORGANIZAR EL DATASET
# ===========================================
import shutil, zipfile, os

# Limpiar carpetas previas antes de preparar un nuevo dataset.
for p in [DATASET_RAW_DIR, DATASET_DIR]:
    if p.exists():
        shutil.rmtree(p)

DATASET_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
ANN_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(DATASET_RAW_DIR)

print("Contenido extraido en:", DATASET_RAW_DIR)

# Buscar archivos COCO e imagenes dentro del ZIP.
json_files = list(DATASET_RAW_DIR.rglob("*.json"))
image_files = []
for ext in ("*.jpg", "*.jpeg", "*.png"):
    image_files.extend(DATASET_RAW_DIR.rglob(ext))

if not json_files:
    raise FileNotFoundError("No se encontraron archivos .json en el ZIP")
if not image_files:
    raise FileNotFoundError("No se encontraron imagenes en el ZIP")

print("JSON encontrados:")
for jf in json_files:
    print("-", jf.relative_to(DATASET_RAW_DIR))

# Buscar JSON de entrenamiento y validacion.
train_json = None
val_json = None

for jf in json_files:
    rel_path = str(jf.relative_to(DATASET_RAW_DIR)).lower().replace("\", "/")
    name = jf.name.lower()

    if train_json is None and ("/train/" in f"/{rel_path}" or "train" in name):
        train_json = jf
    elif val_json is None and (("/valid/" in f"/{rel_path}") or ("/val/" in f"/{rel_path}") or ("valid" in name) or ("val" in name)):
        val_json = jf

if train_json is None:
    train_json = json_files[0]
    print("Aviso: no se detecto un JSON de train por nombre o carpeta. Se usara el primer JSON encontrado.")

if val_json is None:
    val_json = train_json
    print("Aviso: no se detecto un JSON de validacion. Se reutilizara el JSON de train.")

# Crear la estructura COCO que usara YOLOX dentro de Colab.
(DATASET_DIR / "train2017").mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "val2017").mkdir(parents=True, exist_ok=True)

# Este notebook copia las imagenes encontradas a train2017 y val2017.
# La separacion real entre train y val la define cada archivo JSON COCO.
for img in image_files:
    shutil.copy2(img, DATASET_DIR / "train2017" / img.name)
    shutil.copy2(img, DATASET_DIR / "val2017" / img.name)

shutil.copy2(train_json, ANN_DIR / "instances_train.json")
shutil.copy2(val_json, ANN_DIR / "instances_val.json")

print("Dataset reorganizado en:", DATASET_DIR)
print("JSON train usado:", train_json.relative_to(DATASET_RAW_DIR))
print("JSON val usado  :", val_json.relative_to(DATASET_RAW_DIR))
print("train_json final:", ANN_DIR / "instances_train.json")
print("val_json final  :", ANN_DIR / "instances_val.json")


In [ ]:
# ==================================
# 8) VALIDAR ESTRUCTURA DEL DATASET
# ==================================
import json

train_imgs = list((DATASET_DIR / "train2017").glob("*"))
val_imgs = list((DATASET_DIR / "val2017").glob("*"))

print("Imagenes en train2017:", len(train_imgs))
print("Imagenes en val2017  :", len(val_imgs))
print("Nota: estas carpetas son una reorganizacion interna para YOLOX.")
print("La asignacion real de train y val depende del contenido de los JSON COCO detectados arriba.")

for jf in [ANN_DIR / "instances_train.json", ANN_DIR / "instances_val.json"]:
    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("\nArchivo:", jf.name)
    print("images     =", len(data.get("images", [])))
    print("annotations=", len(data.get("annotations", [])))
    print("categories =", len(data.get("categories", [])))

    if not data.get("images"):
        raise ValueError(f"{jf.name} no contiene la clave 'images' o esta vacia")
    if not data.get("categories"):
        raise ValueError(f"{jf.name} no contiene la clave 'categories' o esta vacia")


In [ ]:
# =======================================
# 9) CREAR yolox_config.py DINAMICO
# =======================================
import json

with open(ANN_DIR / "instances_train.json", "r", encoding="utf-8") as f:
    coco_train = json.load(f)

num_classes = len(coco_train.get("categories", []))
if num_classes <= 0:
    raise ValueError("No se detectaron categorias en instances_train.json")

config_text = f'''
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        self.num_classes = {num_classes}
        self.depth = {depth}
        self.width = {width}
        self.data_dir = "{DATASET_DIR}"
        self.train_ann = "instances_train.json"
        self.val_ann = "instances_val.json"
        self.input_size = ({INPUT_HEIGHT}, {INPUT_WIDTH})
        self.test_size = ({INPUT_HEIGHT}, {INPUT_WIDTH})
        self.multiscale_range = 0
        self.random_size = None
        self.max_epoch = {MAX_EPOCH}
        self.data_num_workers = {NUM_WORKERS}
        self.eval_interval = 1
        self.print_interval = 10
        self.basic_lr_per_img = 0.01 / 64.0
        self.warmup_epochs = 1
        self.no_aug_epochs = 1
        self.enable_mixup = False
        self.mosaic_prob = 1.0
        self.mixup_prob = 0.0
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.exp_name = "yolox_custom_coco"

def get_exp():
    return Exp()
'''

EXP_FILE.write_text(config_text, encoding="utf-8")
print("Archivo creado:", EXP_FILE)
print(EXP_FILE.read_text(encoding="utf-8"))


In [ ]:
# ======================================
# 10) ENTRENAR EL MODELO EN YOLOX
# ======================================
import os, torch
%cd /content/YOLOX

if not torch.cuda.is_available():
    raise RuntimeError(
        "Torch no tiene CUDA disponible. Revisa que Colab tenga GPU activa y reinicia el entorno."
    )

os.environ["PYTHONPATH"] = f"/content/YOLOX:" + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH usado para entrenamiento:", os.environ["PYTHONPATH"])
print("torch.cuda.is_available() =", torch.cuda.is_available())
print("GPU actual:", torch.cuda.get_device_name(0))

train_cmd = f"PYTHONPATH=/content/YOLOX:$PYTHONPATH python tools/train.py -f {EXP_FILE} -d 1 -b {BATCH_SIZE} -o "
if USE_FP16:
    train_cmd += f"--fp16 -c {PRETRAINED_PATH}"
else:
    train_cmd += f"-c {PRETRAINED_PATH}"

print("Comando de entrenamiento:")
print(train_cmd)

!{train_cmd}


In [ ]:
# ==========================================
# 11) ENCONTRAR EL CHECKPOINT ENTRENADO
# ==========================================
from pathlib import Path

output_dir = BASE_DIR / "YOLOX" / "YOLOX_outputs" / "yolox_custom_coco"

candidates = [
    output_dir / "best_ckpt.pth",
    output_dir / "latest_ckpt.pth",
]

trained_ckpt = None
for c in candidates:
    if c.exists():
        trained_ckpt = c
        break

if trained_ckpt is None:
    all_pth = list(output_dir.rglob("*.pth")) if output_dir.exists() else []
    raise FileNotFoundError(f"No se encontro checkpoint entrenado en {output_dir}. Encontrados: {all_pth}")

print("Checkpoint encontrado:", trained_ckpt)


In [ ]:
# ==========================================================
# 12) VALIDAR DEPENDENCIAS NECESARIAS ANTES DE EXPORTAR A ONNX
# ==========================================================
import onnx, onnxscript, onnxruntime

print("onnx       =", onnx.__version__)
print("onnxscript =", onnxscript.__version__)
print("onnxruntime=", onnxruntime.__version__)


In [ ]:
# =========================================================
# 13) PARCHEAR tools/export_onnx.py PARA COLAB ACTUAL
# =========================================================
from pathlib import Path

export_file = Path("/content/YOLOX/tools/export_onnx.py")
txt = export_file.read_text(encoding="utf-8")

# Parche 1: PyTorch 2.6+ cambio el default de weights_only en torch.load
txt = txt.replace(
    'ckpt = torch.load(ckpt_file, map_location="cpu")',
    'ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)'
)

# Parche 2: torch.onnx._export ya no existe en PyTorch reciente
txt = txt.replace(
    "torch.onnx._export(",
    "torch.onnx.export("
)

export_file.write_text(txt, encoding="utf-8")
print("Parche aplicado en:", export_file)


In [ ]:
# =========================================
# 14) EXPORTAR EL CHECKPOINT A ONNX
# =========================================
%cd /content/YOLOX

from pathlib import Path
import subprocess
import os

onnx_path = output_dir / "best_ckpt_empaquetado.onnx"

cmd = [
    "python", "tools/export_onnx.py",
    "-f", str(EXP_FILE),
    "-c", str(trained_ckpt),
    "--output-name", str(onnx_path),
    "--no-onnxsim",
    "--decode_in_inference",
]

env = os.environ.copy()
env["PYTHONPATH"] = "/content/YOLOX:" + env.get("PYTHONPATH", "")

print("Comando real:")
print("PYTHONPATH=" + env["PYTHONPATH"])
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    cwd="/content/YOLOX",
    env=env,
    capture_output=True,
    text=True
)

print("\n===== STDOUT =====")
print(result.stdout if result.stdout else "(vacio)")

print("\n===== STDERR =====")
print(result.stderr if result.stderr else "(vacio)")

print("\nCodigo de salida:", result.returncode)
print("Existe el ONNX?:", onnx_path.exists(), onnx_path)

if not onnx_path.exists():
    raise FileNotFoundError(f"No se genero el ONNX en {onnx_path}")

print("Modelo ONNX generado en:", onnx_path)


In [ ]:
# ===================================================
# 15) INSERTAR LAS CLASES EN EL METADATO DEL ONNX
# ===================================================
import json
import onnx
import uuid

# Se genera un nombre unico para que el usuario descargue un ONNX final identificado.
random_id = uuid.uuid4().hex[:8]

onnx_in = output_dir / "best_ckpt_empaquetado.onnx"
onnx_out = output_dir / f"best_ckpt_{random_id}.onnx"

with open(ANN_DIR / "instances_train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

class_dict = {idx: cat["name"] for idx, cat in enumerate(data.get("categories", []))}

model = onnx.load(str(onnx_in))

# Se reemplaza el metadata previo 'names' si ya existiera.
kept = [p for p in model.metadata_props if p.key != "names"]
del model.metadata_props[:]
model.metadata_props.extend(kept)

model.metadata_props.append(
    onnx.StringStringEntryProto(
        key="names",
        value=json.dumps(class_dict, ensure_ascii=False)
    )
)

onnx.save(model, str(onnx_out))

print("ONNX final con metadato 'names' guardado en:", onnx_out)
print("Las clases embebidas en el modelo son:", class_dict)


In [ ]:
# ============================================
# 16) VERIFICAR METADATO DEL ONNX GENERADO
# ============================================
import onnx, json

m = onnx.load(str(output_dir / f"best_ckpt_{random_id}.onnx"))
meta = {p.key: p.value for p in m.metadata_props}

print("Metadata encontrada:")
for k, v in meta.items():
    print(f"{k} = {v}")

if "names" not in meta:
    raise KeyError("No se encontro el metadato 'names' en el ONNX final")

names = json.loads(meta["names"])
print("\nClases recuperadas desde metadata:")
print(names)


## Nota

El ONNX final generado por este notebook queda pensado para una inferencia donde la salida ya incluye la decodificacion dentro del modelo exportado.

- Incluye el metadato `names` con los nombres de clase detectados desde `categories`.
- Si luego cambias tu pipeline de inferencia, revisa si tu codigo espera salida decodificada o salida cruda del modelo.
- Si usas otro script distinto para ONNX, valida primero que no aplique un postproceso incompatible.


In [ ]:
# ==================================
# 17) DESCARGAR SOLO EL ONNX FINAL
# ==================================
from google.colab import files

final_onnx = output_dir / f"best_ckpt_{random_id}.onnx"

if not final_onnx.exists():
    raise FileNotFoundError(f"No existe el archivo final esperado: {final_onnx}")

print("Descargando:", final_onnx)
print("Este es el archivo final que debes conservar y usar fuera de Colab.")
files.download(str(final_onnx))
